# Árvore AVL: teoria, algoritmos e prática

Este notebook apresenta a **estrutura de dados Árvore AVL** em português, com explicações e implementações dos algoritmos principais.

Objetivos:
- Entender o que é uma AVL e por que ela é balanceada.
- Implementar rotações simples e duplas.
- Implementar inserção, remoção e busca.
- Validar invariantes da estrutura com testes.
- Medir desempenho básico em cenários práticos.

In [1]:
# -*- coding: utf-8 -*-
"""
Seção 1: Configuração do notebook (UTF-8) e imports.
"""

import random
import time

print("Validação de acentuação: árvore, inserção, remoção, equilíbrio, rotação.")

Validação de acentuação: árvore, inserção, remoção, equilíbrio, rotação.


## 2) Estrutura do nó AVL

Cada nó armazena:
- `chave`: valor usado para ordenação.
- `esq` e `dir`: ponteiros para filhos.
- `altura`: altura do nó na árvore.

Armazenar a altura permite recalcular o balanceamento local e manter operações com custo esperado de $O(\log n)$ em árvores balanceadas.

In [2]:
class NoAVL:
    def __init__(self, chave, esq=None, dir=None, altura=1):
        self.chave = chave
        self.esq = esq
        self.dir = dir
        self.altura = altura

## 3) Altura e fator de balanceamento

Definimos o fator de balanceamento de um nó $n$ por:

$$
fb(n) = h(\text{esq}) - h(\text{dir})
$$

Em uma AVL válida, para todo nó, deve valer $|fb(n)| \le 1$.

In [3]:
def altura(no):
    return no.altura if no else 0


def atualizar_altura(no):
    no.altura = 1 + max(altura(no.esq), altura(no.dir))


def fator_balanceamento(no):
    if no is None:
        return 0
    return altura(no.esq) - altura(no.dir)


# Exemplo simples de cálculo de fb
raiz_exemplo = NoAVL(10, NoAVL(5), NoAVL(15))
atualizar_altura(raiz_exemplo.esq)
atualizar_altura(raiz_exemplo.dir)
atualizar_altura(raiz_exemplo)
print("fb(10) =", fator_balanceamento(raiz_exemplo))

fb(10) = 0


## 4) Rotações simples (LL e RR)

Quando um nó fica desequilibrado, aplicamos rotações para restaurar $|fb| \le 1$.

- **Rotação à direita (caso LL):** excesso à esquerda.
- **Rotação à esquerda (caso RR):** excesso à direita.

In [4]:
def rotacao_direita(no_desbalanceado):
    filho_esq = no_desbalanceado.esq
    subarvore_transferida = filho_esq.dir if filho_esq else None

    filho_esq.dir = no_desbalanceado
    no_desbalanceado.esq = subarvore_transferida

    atualizar_altura(no_desbalanceado)
    atualizar_altura(filho_esq)
    return filho_esq


def rotacao_esquerda(no_desbalanceado):
    filho_dir = no_desbalanceado.dir
    subarvore_transferida = filho_dir.esq if filho_dir else None

    filho_dir.esq = no_desbalanceado
    no_desbalanceado.dir = subarvore_transferida

    atualizar_altura(no_desbalanceado)
    atualizar_altura(filho_dir)
    return filho_dir


def imprimir_arvore(no, nivel=0, prefixo="Raiz: "):
    if no is not None:
        print(" " * (4 * nivel) + prefixo + f"{no.chave} (h={no.altura}, fb={fator_balanceamento(no)})")
        imprimir_arvore(no.esq, nivel + 1, "E--- ")
        imprimir_arvore(no.dir, nivel + 1, "D--- ")


# Demonstração curta de rotação LL
no_caso_ll = NoAVL(30, NoAVL(20, NoAVL(10)))
atualizar_altura(no_caso_ll.esq.esq)
atualizar_altura(no_caso_ll.esq)
atualizar_altura(no_caso_ll)
print("Antes da rotação à direita:")
imprimir_arvore(no_caso_ll)
no_caso_ll = rotacao_direita(no_caso_ll)
print("\nDepois da rotação à direita:")
imprimir_arvore(no_caso_ll)


Antes da rotação à direita:
Raiz: 30 (h=3, fb=2)
    E--- 20 (h=2, fb=1)
        E--- 10 (h=1, fb=0)

Depois da rotação à direita:
Raiz: 20 (h=2, fb=0)
    E--- 10 (h=1, fb=0)
    D--- 30 (h=1, fb=0)


## 5) Rotações duplas (LR e RL)

Casos duplos:
- **LR:** o nó está pesado à esquerda, mas o filho esquerdo está pesado à direita.
- **RL:** o nó está pesado à direita, mas o filho direito está pesado à esquerda.

Esses casos são resolvidos combinando duas rotações simples.

In [5]:
def rebalancear(no):
    fator_bal = fator_balanceamento(no)

    # LL: pesado à esquerda e filho esquerdo também à esquerda (ou equilibrado)
    if fator_bal > 1 and fator_balanceamento(no.esq) >= 0:
        return rotacao_direita(no)

    # RR: pesado à direita e filho direito também à direita (ou equilibrado)
    if fator_bal < -1 and fator_balanceamento(no.dir) <= 0:
        return rotacao_esquerda(no)

    # LR: pesado à esquerda, mas filho esquerdo pesado à direita
    if fator_bal > 1 and fator_balanceamento(no.esq) < 0:
        no.esq = rotacao_esquerda(no.esq)
        return rotacao_direita(no)

    # RL: pesado à direita, mas filho direito pesado à esquerda
    if fator_bal < -1 and fator_balanceamento(no.dir) > 0:
        no.dir = rotacao_direita(no.dir)
        return rotacao_esquerda(no)

    return no


# Demonstração curta de caso LR
no_caso_lr = NoAVL(30, NoAVL(10, None, NoAVL(20)))
atualizar_altura(no_caso_lr.esq.dir)
atualizar_altura(no_caso_lr.esq)
atualizar_altura(no_caso_lr)
print("Antes do rebalanceamento LR:")
imprimir_arvore(no_caso_lr)
no_caso_lr = rebalancear(no_caso_lr)
print("\nDepois do rebalanceamento LR:")
imprimir_arvore(no_caso_lr)


Antes do rebalanceamento LR:
Raiz: 30 (h=3, fb=2)
    E--- 10 (h=2, fb=-1)
        D--- 20 (h=1, fb=0)

Depois do rebalanceamento LR:
Raiz: 20 (h=2, fb=0)
    E--- 10 (h=1, fb=0)
    D--- 30 (h=1, fb=0)


## 6) Inserção balanceada em AVL

A inserção em AVL segue dois passos:
1. Inserir como em uma BST comum.
2. Subir na recursão atualizando alturas e rebalanceando com os 4 casos (LL, RR, LR, RL).

In [6]:
def inserir(no, chave):
    if no is None:
        return NoAVL(chave)

    if chave < no.chave:
        no.esq = inserir(no.esq, chave)
    elif chave > no.chave:
        no.dir = inserir(no.dir, chave)
    else:
        return no  # Ignora duplicatas

    atualizar_altura(no)
    return rebalancear(no)


# Exibir sequência de inserções e estado da árvore
sequencia = [10, 20, 30, 40, 50, 25]
raiz = None
for valor in sequencia:
    raiz = inserir(raiz, valor)
    print(f"\nApós inserir {valor}:")
    imprimir_arvore(raiz)


Após inserir 10:
Raiz: 10 (h=1, fb=0)

Após inserir 20:
Raiz: 10 (h=2, fb=-1)
    D--- 20 (h=1, fb=0)

Após inserir 30:
Raiz: 20 (h=2, fb=0)
    E--- 10 (h=1, fb=0)
    D--- 30 (h=1, fb=0)

Após inserir 40:
Raiz: 20 (h=3, fb=-1)
    E--- 10 (h=1, fb=0)
    D--- 30 (h=2, fb=-1)
        D--- 40 (h=1, fb=0)

Após inserir 50:
Raiz: 20 (h=3, fb=-1)
    E--- 10 (h=1, fb=0)
    D--- 40 (h=2, fb=0)
        E--- 30 (h=1, fb=0)
        D--- 50 (h=1, fb=0)

Após inserir 25:
Raiz: 30 (h=3, fb=0)
    E--- 20 (h=2, fb=0)
        E--- 10 (h=1, fb=0)
        D--- 25 (h=1, fb=0)
    D--- 40 (h=2, fb=-1)
        D--- 50 (h=1, fb=0)


## 7) Remoção balanceada em AVL

Na remoção, tratamos três casos:
- Nó folha.
- Nó com um filho.
- Nó com dois filhos (substituição pelo sucessor em-ordem).

Depois da remoção BST, atualizamos altura e rebalanceamos.

In [7]:
def no_minimo(no):
    atual = no
    while atual.esq is not None:
        atual = atual.esq
    return atual


def remover(no, chave):
    if no is None:
        return None

    if chave < no.chave:
        no.esq = remover(no.esq, chave)
    elif chave > no.chave:
        no.dir = remover(no.dir, chave)
    else:
        if no.esq is None:
            return no.dir
        if no.dir is None:
            return no.esq

        sucessor = no_minimo(no.dir)
        no.chave = sucessor.chave
        no.dir = remover(no.dir, sucessor.chave)

    atualizar_altura(no)
    return rebalancear(no)


# Demonstrações de remoção
for valor_remover in [40, 50, 30]:
    raiz = remover(raiz, valor_remover)
    print(f"\nApós remover {valor_remover}:")
    imprimir_arvore(raiz)


Após remover 40:
Raiz: 30 (h=3, fb=1)
    E--- 20 (h=2, fb=0)
        E--- 10 (h=1, fb=0)
        D--- 25 (h=1, fb=0)
    D--- 50 (h=1, fb=0)

Após remover 50:
Raiz: 20 (h=3, fb=-1)
    E--- 10 (h=1, fb=0)
    D--- 30 (h=2, fb=1)
        E--- 25 (h=1, fb=0)

Após remover 30:
Raiz: 20 (h=2, fb=0)
    E--- 10 (h=1, fb=0)
    D--- 25 (h=1, fb=0)


## 8) Busca e percursos (pré, em e pós-ordem)

A busca em AVL segue a mesma lógica de BST, com profundidade típica de $O(\log n)$ em árvores balanceadas.

In [8]:
def buscar(no, chave):
    if no is None or no.chave == chave:
        return no
    if chave < no.chave:
        return buscar(no.esq, chave)
    return buscar(no.dir, chave)


def em_ordem(no):
    if no is None:
        return []
    return em_ordem(no.esq) + [no.chave] + em_ordem(no.dir)


def pre_ordem(no):
    if no is None:
        return []
    return [no.chave] + pre_ordem(no.esq) + pre_ordem(no.dir)


def pos_ordem(no):
    if no is None:
        return []
    return pos_ordem(no.esq) + pos_ordem(no.dir) + [no.chave]

print("Em ordem:", em_ordem(raiz))
print("Pré-ordem:", pre_ordem(raiz))
print("Pós-ordem:", pos_ordem(raiz))
print("Buscar 25:", "encontrado" if buscar(raiz, 25) else "não encontrado")

Em ordem: [10, 20, 25]
Pré-ordem: [20, 10, 25]
Pós-ordem: [10, 25, 20]
Buscar 25: encontrado


## 9) Validar invariantes da AVL com testes automatizados

Vamos verificar automaticamente:
- Propriedade BST.
- Consistência das alturas armazenadas.
- Restrição AVL: $|fb(n)| \le 1$ em todos os nós.

In [9]:
def eh_bst(no, minimo=-10**18, maximo=10**18):
    if no is None:
        return True
    if not (minimo < no.chave < maximo):
        return False
    return eh_bst(no.esq, minimo, no.chave) and eh_bst(no.dir, no.chave, maximo)


def alturas_consistentes(no):
    if no is None:
        return True, 0

    ok_esq, h_esq = alturas_consistentes(no.esq)
    ok_dir, h_dir = alturas_consistentes(no.dir)
    h_calc = 1 + max(h_esq, h_dir)

    return ok_esq and ok_dir and (no.altura == h_calc), h_calc


def eh_avl(no):
    if no is None:
        return True
    fb = fator_balanceamento(no)
    if abs(fb) > 1:
        return False
    return eh_avl(no.esq) and eh_avl(no.dir)


random.seed(7)
dados_teste = random.sample(range(1, 5000), 500)
raiz_teste = None
for x in dados_teste:
    raiz_teste = inserir(raiz_teste, x)

remocoes_teste = random.sample(dados_teste, 120)
for x in remocoes_teste:
    raiz_teste = remover(raiz_teste, x)

ok_alturas, _ = alturas_consistentes(raiz_teste)
assert eh_bst(raiz_teste), "Violação da propriedade BST"
assert ok_alturas, "Alturas inconsistentes"
assert eh_avl(raiz_teste), "Violação do balanceamento AVL"
print("Testes concluídos com sucesso: invariantes preservadas.")

Testes concluídos com sucesso: invariantes preservadas.


## 10) Cenários práticos e desempenho básico

Vamos comparar tempos médios de inserção, busca e remoção entre:
- AVL (balanceada automaticamente).
- BST simples (sem balanceamento).

Observação: este experimento é didático e os resultados variam conforme máquina e distribuição dos dados.

In [10]:
class NoBST:
    def __init__(self, chave, esq=None, dir=None):
        self.chave = chave
        self.esq = esq
        self.dir = dir


def bst_inserir(no, chave):
    if no is None:
        return NoBST(chave)
    if chave < no.chave:
        no.esq = bst_inserir(no.esq, chave)
    elif chave > no.chave:
        no.dir = bst_inserir(no.dir, chave)
    return no


def bst_buscar(no, chave):
    if no is None or no.chave == chave:
        return no
    if chave < no.chave:
        return bst_buscar(no.esq, chave)
    return bst_buscar(no.dir, chave)


def bst_minimo(no):
    while no.esq:
        no = no.esq
    return no


def bst_remover(no, chave):
    if no is None:
        return None
    if chave < no.chave:
        no.esq = bst_remover(no.esq, chave)
    elif chave > no.chave:
        no.dir = bst_remover(no.dir, chave)
    else:
        if no.esq is None:
            return no.dir
        if no.dir is None:
            return no.esq
        sucessor = bst_minimo(no.dir)
        no.chave = sucessor.chave
        no.dir = bst_remover(no.dir, sucessor.chave)
    return no


def medir_avl(valores, consultas, remocoes):
    raiz_local = None

    tempo_inicio = time.perf_counter()
    for v in valores:
        raiz_local = inserir(raiz_local, v)
    tempo_insercao = time.perf_counter() - tempo_inicio

    tempo_inicio = time.perf_counter()
    for chave_consulta in consultas:
        _ = buscar(raiz_local, chave_consulta)
    tempo_busca = time.perf_counter() - tempo_inicio

    tempo_inicio = time.perf_counter()
    for chave_remocao in remocoes:
        raiz_local = remover(raiz_local, chave_remocao)
    tempo_remocao = time.perf_counter() - tempo_inicio

    return tempo_insercao, tempo_busca, tempo_remocao


def medir_bst(valores, consultas, remocoes):
    raiz_local = None

    tempo_inicio = time.perf_counter()
    for v in valores:
        raiz_local = bst_inserir(raiz_local, v)
    tempo_insercao = time.perf_counter() - tempo_inicio

    tempo_inicio = time.perf_counter()
    for chave_consulta in consultas:
        _ = bst_buscar(raiz_local, chave_consulta)
    tempo_busca = time.perf_counter() - tempo_inicio

    tempo_inicio = time.perf_counter()
    for chave_remocao in remocoes:
        raiz_local = bst_remover(raiz_local, chave_remocao)
    tempo_remocao = time.perf_counter() - tempo_inicio

    return tempo_insercao, tempo_busca, tempo_remocao


n = 3000
dados = random.sample(range(1, 10 * n), n)
consultas = random.sample(dados, 1000)
remocoes = random.sample(dados, 600)

avl_tempo_insercao, avl_tempo_busca, avl_tempo_remocao = medir_avl(dados, consultas, remocoes)
bst_tempo_insercao, bst_tempo_busca, bst_tempo_remocao = medir_bst(dados, consultas, remocoes)

print("Resultados (segundos):")
print(f"AVL -> inserção: {avl_tempo_insercao:.6f}, busca: {avl_tempo_busca:.6f}, remoção: {avl_tempo_remocao:.6f}")
print(f"BST -> inserção: {bst_tempo_insercao:.6f}, busca: {bst_tempo_busca:.6f}, remoção: {bst_tempo_remocao:.6f}")


Resultados (segundos):
AVL -> inserção: 0.009996, busca: 0.000437, remoção: 0.001712
BST -> inserção: 0.001860, busca: 0.000511, remoção: 0.000359
